## Summary: End-to-End WaterWatch Pipeline

This notebook demonstrates the complete WaterWatch pipeline:

1. **Authentication** ✓ OAuth2 to Copernicus Data Space
2. **Catalog Search** ✓ Sentinel-2 L2A and Sentinel-1 GRD discovery
3. **Optical Data** ✓ NDWI water index calculation
4. **Radar Data** ✓ SAR VV backscatter (cloud fallback)
5. **Data Fusion** ✓ Z-score anomaly detection
6. **Citizen Integration** ✓ Crowd-sourced observations
7. **Flow Modeling** ✓ Downstream ETA prediction
8. **Alert JSON** ✓ Production-ready output

### Key Features

- **Scientific Responsibility**: Uses "satellite-based anomaly", "risk estimation", avoids false certainty
- **Multi-Source Fusion**: Combines optical + radar for robustness
- **Citizen Crowdsourcing**: Integrates community reports for enhanced confidence
- **Downstream Propagation**: Models flow to downstream monitoring points
- **MVP-Ready**: Hardcoded AOIs (Vardar, Treska, Ohrid), hackathon-demo code

### Next Steps

1. **Deploy Backend**: Run `backend/app.py` with real Copernicus credentials
2. **Build Frontend**: Use `frontend/src/api/waterwatchApi.js` client
3. **Scale Data**: Replace simulated data with real Process API responses
4. **Add Database**: Persist citizen reports and historical trends
5. **Test & Validate**: Cross-validate with ground observations

### References

- [Copernicus Data Space](https://dataspace.copernicus.eu/)
- [Sentinel Hub API Docs](https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/)
- [NDWI Literature](https://en.wikipedia.org/wiki/Normalized_difference_water_index)
- [SAR Water Detection](https://www.usgs.gov/faqs/what-difference-between-sar-and-optical-imagery)

In [ ]:
from enum import Enum
import uuid

class AlertType(Enum):
    """Alert severity levels."""
    NORMAL = "normal"
    WARNING = "warning"
    CRITICAL = "critical"
    DATA_SPARSE = "data_sparse"

def classify_alert_type(
    fused_sigma: float,
    confidence: float,
    citizen_report_count: int = 0,
) -> AlertType:
    """Classify alert severity."""
    if confidence < 0.4:
        return AlertType.DATA_SPARSE
    
    anomaly_score = abs(fused_sigma) * confidence
    citizen_factor = min(1.0, citizen_report_count / 3.0)
    total_score = (anomaly_score * 0.7) + (citizen_factor * 0.3)
    
    if total_score > 2.0:
        return AlertType.CRITICAL
    elif total_score > 1.0:
        return AlertType.WARNING
    else:
        return AlertType.NORMAL

def generate_alert_message(alert_type: AlertType, station_name: str, fused_result: Dict) -> str:
    """Generate responsible scientific message."""
    if alert_type == AlertType.DATA_SPARSE:
        return f"⚠️ INSUFFICIENT DATA - {station_name}\nCannot generate reliable early warning at this time."
    
    elif alert_type == AlertType.CRITICAL:
        return (
            f"🚨 CRITICAL ALERT - {station_name}\n"
            f"Satellite-based anomaly detected (confidence: {fused_result['confidence']:.0%}).\n"
            f"Possible pollution risk or environmental change indicated by {fused_result['source']}.\n"
            f"RECOMMEND: Immediate on-site verification and monitoring.\n"
            f"⚠️ This is an early warning system - professional assessment required."
        )
    
    elif alert_type == AlertType.WARNING:
        return (
            f"⚠️ WARNING - {station_name}\n"
            f"Satellite-based risk estimation elevated (confidence: {fused_result['confidence']:.0%}).\n"
            f"Anomaly detected via {fused_result['source']}. Recommend increased monitoring.\n"
            f"Cross-reference with ground observations and citizen reports."
        )
    
    else:
        return f"✓ {station_name} appears normal based on satellite observation."

def generate_recommended_action(alert_type: AlertType) -> str:
    """Generate actionable recommendations."""
    if alert_type == AlertType.CRITICAL:
        return (
            "IMMEDIATE: 1) Verify with on-site data. 2) Notify downstream users. "
            "3) Increase monitoring. 4) Prepare response protocols."
        )
    elif alert_type == AlertType.WARNING:
        return (
            "MONITOR: 1) Increase satellite frequency. 2) Coordinate with sensors. "
            "3) Prepare response. 4) Document baseline."
        )
    elif alert_type == AlertType.DATA_SPARSE:
        return (
            "DATA LIMITED: 1) Await next satellite pass. 2) Use alternative sensors. "
            "3) Rely on citizen observers."
        )
    else:
        return "Continue routine monitoring. No unusual signals detected."

# Generate alert
print("\n" + "="*70)
print("ALERT GENERATION")
print("="*70)

alert_type = classify_alert_type(
    fused_result["fused_sigma"],
    fused_result["confidence"],
    len(CITIZEN_REPORTS),
)

print(f"\n🔔 Alert Type: {alert_type.value.upper()}")

# Prepare final alert JSON
alert_json = {
    "alert_id": f"alert_{uuid.uuid4().hex[:12]}",
    "timestamp_utc": datetime.utcnow().isoformat() + "Z",
    "expires_at": (datetime.utcnow() + timedelta(hours=6)).isoformat() + "Z",
    
    "station": {
        "id": ACTIVE_STATION_ID,
        "name": STATION_CONFIG["name"],
        "river": STATION_CONFIG["river"],
        "segment": STATION_CONFIG["segment"],
        "bbox": STATION_CONFIG["bbox"],
    },
    
    "alert": {
        "type": alert_type.value,
        "message": generate_alert_message(alert_type, STATION_CONFIG["name"], fused_result),
        "recommended_action": generate_recommended_action(alert_type),
        "confidence": fused_result["confidence"],
    },
    
    "satellite_signal": {
        "source": fused_result["source"],
        "fused_sigma": fused_result["fused_sigma"],
        "anomaly_detected": fused_result["anomaly_detected"],
        "ndwi": fused_result["ndwi_contribution"],
        "sar": fused_result["sar_contribution"],
    },
    
    "citizen_reports": reports_agg,
    
    "downstream_movement": {
        "targets": downstream_targets if downstream_targets else [],
        "flow_speed_mps": STATION_CONFIG["flow_speed_mps"],
    },
}

# Output alert JSON
print("\n" + "="*70)
print("FINAL WATERWATCHALERT JSON OUTPUT")
print("="*70)
print(json.dumps(alert_json, indent=2))

# Save to file
output_file = "waterwatchalert_output.json"
with open(output_file, "w") as f:
    json.dump(alert_json, f, indent=2)

print(f"\n✓ Alert saved to {output_file}")

## Section 10: Alert JSON Generation and Output

Construct production-ready alert JSON with satellite signals, citizen reports, movement model,
and recommended actions for downstream monitoring.

In [ ]:
@dataclass
class MovementModel:
    """Downstream movement prediction."""
    origin_station_id: str
    target_location: str
    distance_meters: float
    flow_speed_mps: float
    estimated_arrival_minutes: float
    confidence: float
    
    def to_dict(self) -> Dict:
        return asdict(self)

# Pre-defined downstream distances (in real system: use GIS)
DOWNSTREAM_DISTANCES = {
    ("vardar_vs", "skopje_city"): 15000,
    ("vardar_vs", "vardar_mouth"): 85000,
    ("treska_km", "vardar_vs"): 42000,
    ("ohrid_s", "ohrid_city"): 18000,
}

def estimate_downstream_arrival(
    origin_station_id: str,
    target_station_id: str,
    distance_meters: float,
    flow_speed_mps: float,
) -> MovementModel:
    """Estimate anomaly arrival time at downstream station."""
    
    arrival_minutes = (distance_meters / flow_speed_mps) / 60.0
    
    # Confidence decreases with distance (model uncertainty grows)
    confidence = max(0.3, 1.0 - (distance_meters / 100000.0))
    
    return MovementModel(
        origin_station_id=origin_station_id,
        target_location=target_station_id,
        distance_meters=distance_meters,
        flow_speed_mps=flow_speed_mps,
        estimated_arrival_minutes=arrival_minutes,
        confidence=confidence,
    )

def get_actionable_targets(
    origin_station_id: str,
    hours_ahead: int = 24,
) -> List[Dict[str, Any]]:
    """Get downstream targets within time horizon."""
    targets = []
    downstream = STATION_CONFIG.get("downstream_targets", [])
    
    for target_id in downstream:
        key = (origin_station_id, target_id)
        distance = DOWNSTREAM_DISTANCES.get(key, 30000)
        flow_speed = STATION_CONFIG.get("flow_speed_mps", 1.0)
        
        model = estimate_downstream_arrival(
            origin_station_id, target_id, distance, flow_speed
        )
        
        if model.estimated_arrival_minutes <= (hours_ahead * 60):
            targets.append({
                "target_id": target_id,
                "distance_meters": distance,
                "arrival_minutes": model.estimated_arrival_minutes,
                "arrival_datetime": (
                    datetime.utcnow() + timedelta(minutes=model.estimated_arrival_minutes)
                ).isoformat() + "Z",
                "confidence": model.confidence,
            })
    
    return targets

# Demo: Calculate downstream arrivals
print("\n" + "="*70)
print("DOWNSTREAM MOVEMENT MODELING")
print("="*70)

downstream_targets = get_actionable_targets(ACTIVE_STATION_ID, hours_ahead=24)

print(f"\n🌊 Downstream Targets from {STATION_CONFIG['name']}:")
print(f"   Flow Speed: {STATION_CONFIG['flow_speed_mps']} m/s")

if downstream_targets:
    for target in downstream_targets:
        arrival_dt = datetime.fromisoformat(target['arrival_datetime'].replace('Z', '+00:00'))
        hours = int(target['arrival_minutes'] // 60)
        mins = int(target['arrival_minutes'] % 60)
        
        print(f"\n  📍 {target['target_id']}")
        print(f"     Distance: {target['distance_meters']:,} m ({target['distance_meters']/1000:.1f} km)")
        print(f"     ETA: {hours}h {mins}m (arrives ~{arrival_dt.strftime('%H:%M UTC')})")
        print(f"     Confidence: {target['confidence']:.1%}")
else:
    print("\n  ℹ️ No downstream targets within 24h horizon")

## Section 9: Downstream Flow Modeling and ETA Prediction

Estimate downstream propagation of detected anomalies.
Calculate arrival_minutes = river_distance_meters / flow_speed_mps / 60

In [ ]:
from dataclasses import dataclass, asdict
from typing import List

@dataclass
class CitizenReport:
    """Citizen-submitted observation."""
    id: str
    station_id: str
    timestamp: str
    location: Dict[str, float]  # {"lat": float, "lon": float}
    observation_type: str  # "photo", "voice", "description"
    message: str
    confirmed: bool = False
    upvotes: int = 0
    
    def to_dict(self) -> Dict:
        return asdict(self)

# Sample citizen reports for demo
CITIZEN_REPORTS = [
    CitizenReport(
        id="report_001",
        station_id="vardar_vs",
        timestamp="2024-01-15T09:30:00Z",
        location={"lat": 41.97, "lon": 21.55},
        observation_type="photo",
        message="Unusual discoloration observed near Veles dam",
        confirmed=True,
        upvotes=3,
    ),
    CitizenReport(
        id="report_002",
        station_id="vardar_vs",
        timestamp="2024-01-15T08:15:00Z",
        location={"lat": 41.96, "lon": 21.54},
        observation_type="description",
        message="Strong smell of chemicals near the river",
        confirmed=False,
        upvotes=1,
    ),
    CitizenReport(
        id="report_003",
        station_id="vardar_vs",
        timestamp="2024-01-14T18:45:00Z",
        location={"lat": 41.98, "lon": 21.56},
        observation_type="photo",
        message="Dead fish observed downstream",
        confirmed=True,
        upvotes=5,
    ),
]

def aggregate_citizen_reports(reports: List[CitizenReport], station_id: str) -> Dict[str, Any]:
    """Aggregate citizen reports for a station."""
    station_reports = [r for r in reports if r.station_id == station_id]
    confirmed = sum(1 for r in station_reports if r.confirmed)
    total_upvotes = sum(r.upvotes for r in station_reports)
    
    return {
        "report_count": len(station_reports),
        "confirmed_reports": confirmed,
        "total_upvotes": total_upvotes,
        "reports": [r.to_dict() for r in station_reports[-5:]],  # Last 5
    }

def calculate_citizen_confidence_boost(reports: List[CitizenReport], station_id: str) -> float:
    """
    Boost satellite confidence score if citizen reports corroborate anomaly.
    Returns boost factor (1.0 = no boost, up to 1.3 = strong corroboration)
    """
    station_reports = [r for r in reports if r.station_id == station_id]
    
    if not station_reports:
        return 1.0
    
    recent_reports = [r for r in station_reports 
                     if datetime.fromisoformat(r.timestamp.replace('Z', '+00:00')) > 
                        datetime.utcnow() - timedelta(days=1)]
    
    confirmed = sum(1 for r in recent_reports if r.confirmed)
    upvotes = sum(r.upvotes for r in recent_reports)
    
    # Boost based on confirmed reports and community agreement
    boost = 1.0 + (confirmed * 0.08) + (min(upvotes, 5) * 0.02)
    return min(boost, 1.3)  # Cap at 1.3x

# Demo: Aggregate and boost
print("\n" + "="*70)
print("CITIZEN REPORTS")
print("="*70)

reports_agg = aggregate_citizen_reports(CITIZEN_REPORTS, ACTIVE_STATION_ID)
print(f"\n👥 Citizen Reports for {STATION_CONFIG['name']}:")
print(f"  Total Reports: {reports_agg['report_count']}")
print(f"  Confirmed: {reports_agg['confirmed_reports']}")
print(f"  Community Upvotes: {reports_agg['total_upvotes']}")

for report in reports_agg['reports']:
    print(f"\n  • {report['observation_type'].upper()}")
    print(f"    Time: {report['timestamp']}")
    print(f"    Location: {report['location']}")
    print(f"    Message: {report['message']}")
    print(f"    Confirmed: {'✓' if report['confirmed'] else '✗'}, Upvotes: {report['upvotes']}")

# Calculate confidence boost
confidence_boost = calculate_citizen_confidence_boost(CITIZEN_REPORTS, ACTIVE_STATION_ID)
print(f"\n🔼 Confidence Boost Factor: {confidence_boost:.2f}x")

## Section 8: Citizen Reports Integration

Accept and aggregate citizen observations (photos, voice descriptions, locations).
Cross-reference with satellite signals to enhance confidence scoring.

In [ ]:
def analyze_ndwi_anomaly(ndwi_data: np.ndarray, station_config: Dict) -> Dict[str, Any]:
    """Analyze NDWI for water anomalies."""
    baseline = station_config["baseline_ndvi"]
    
    # Focus on water pixels (NDWI > 0.3)
    water_pixels = ndwi_data[ndwi_data > 0.3]
    
    if len(water_pixels) == 0:
        current_value = -0.1
        confidence = 0.3
    else:
        current_value = np.mean(water_pixels)
        confidence = min(1.0, len(water_pixels) / (ndwi_data.size * 0.5))
    
    # Calculate z-score (sigma)
    sigma = (current_value - baseline["mean"]) / (baseline["std"] + 1e-6)
    
    # Flag anomaly if |sigma| > 1.5
    anomaly_detected = abs(sigma) > 1.5
    
    return {
        "source": "sentinel-2-l2a",
        "indicator": "ndwi",
        "current_value": float(current_value),
        "baseline_mean": baseline["mean"],
        "baseline_std": baseline["std"],
        "sigma": float(sigma),
        "confidence": float(confidence),
        "anomaly_detected": bool(anomaly_detected),
        "water_pixel_count": int(len(water_pixels)),
    }

def analyze_sar_anomaly(sar_vv: np.ndarray, station_config: Dict) -> Dict[str, Any]:
    """Analyze SAR VV backscatter for water anomalies."""
    baseline = station_config["baseline_vv_db"]
    
    current_value = np.mean(sar_vv)
    confidence = 0.6  # SAR is weather-independent but lower spatial res
    
    # Calculate z-score
    sigma = (current_value - baseline["mean"]) / (baseline["std"] + 1e-6)
    
    # For SAR: MORE NEGATIVE = MORE WATER-LIKE (smooth surface)
    # Anomaly if significantly LESS negative than baseline (unusual backscatter)
    anomaly_detected = sigma > 1.5
    
    return {
        "source": "sentinel-1-grd",
        "indicator": "vv_backscatter",
        "current_value": float(current_value),
        "baseline_mean": baseline["mean"],
        "baseline_std": baseline["std"],
        "sigma": float(sigma),
        "confidence": float(confidence),
        "anomaly_detected": bool(anomaly_detected),
    }

def fuse_signals(ndwi_result: Dict, sar_result: Dict) -> Dict[str, Any]:
    """
    Fuse NDWI and SAR signals for robust anomaly detection.
    Strategy:
    - If NDWI reliable (conf > 0.5): prioritize NDWI
    - If clouds/no NDWI: use SAR fallback
    - Both available: weighted average
    """
    ndwi_conf = ndwi_result.get("confidence", 0.5)
    sar_conf = sar_result.get("confidence", 0.5)
    
    if ndwi_conf > 0.5:
        if sar_conf > 0.5:
            # Both reliable - weighted fusion
            ndwi_weight, sar_weight = 0.65, 0.35
            fused_sigma = (ndwi_result["sigma"] * ndwi_weight + 
                          sar_result["sigma"] * sar_weight)
            fused_confidence = (ndwi_conf + sar_conf) / 2.0
            source = "fused-s2-s1"
        else:
            # NDWI only
            fused_sigma = ndwi_result["sigma"]
            fused_confidence = ndwi_conf
            source = "sentinel-2-l2a"
    elif sar_conf > 0.4:
        # SAR fallback
        fused_sigma = sar_result["sigma"]
        fused_confidence = sar_conf
        source = "sentinel-1-grd"
    else:
        # No reliable data
        fused_sigma = 0.0
        fused_confidence = 0.0
        source = "none"
    
    anomaly_detected = abs(fused_sigma) > 1.5
    
    return {
        "source": source,
        "fused_sigma": float(fused_sigma),
        "confidence": float(fused_confidence),
        "anomaly_detected": bool(anomaly_detected),
        "ndwi_contribution": ndwi_result,
        "sar_contribution": sar_result,
    }

# Analyze simulated data
print("\n" + "="*70)
print("SATELLITE DATA ANALYSIS")
print("="*70)

ndwi_analysis = analyze_ndwi_anomaly(NDWI_SIMULATED, STATION_CONFIG)
print(f"\n📊 NDWI Analysis:")
print(f"  Current NDWI: {ndwi_analysis['current_value']:.3f}")
print(f"  Baseline: μ={ndwi_analysis['baseline_mean']:.3f}, σ={ndwi_analysis['baseline_std']:.3f}")
print(f"  Z-score (sigma): {ndwi_analysis['sigma']:.2f}")
print(f"  Confidence: {ndwi_analysis['confidence']:.1%}")
print(f"  Anomaly Detected: {'✓ YES' if ndwi_analysis['anomaly_detected'] else '✗ NO'}")

sar_analysis = analyze_sar_anomaly(SAR_VV_SIMULATED, STATION_CONFIG)
print(f"\n📊 SAR VV Backscatter Analysis:")
print(f"  Current VV: {sar_analysis['current_value']:.1f} dB")
print(f"  Baseline: μ={sar_analysis['baseline_mean']:.1f} dB, σ={sar_analysis['baseline_std']:.1f} dB")
print(f"  Z-score (sigma): {sar_analysis['sigma']:.2f}")
print(f"  Confidence: {sar_analysis['confidence']:.1%}")
print(f"  Anomaly Detected: {'✓ YES' if sar_analysis['anomaly_detected'] else '✗ NO'}")

# Fuse signals
fused_result = fuse_signals(ndwi_analysis, sar_analysis)
print(f"\n🔀 FUSED SIGNAL:")
print(f"  Source: {fused_result['source']}")
print(f"  Fused Sigma: {fused_result['fused_sigma']:.2f}")
print(f"  Fused Confidence: {fused_result['confidence']:.1%}")
print(f"  Anomaly Detected: {'✓ YES' if fused_result['anomaly_detected'] else '✗ NO'}")

## Section 7: Data Fusion - Calculate Satellite-Based Risk Signal

Fuse NDWI and SAR signals using statistical z-score approach:
sigma = (current_value - baseline_mean) / baseline_std

Use scientifically responsible language: "satellite-based anomaly", "risk estimation", not "pollution proven".

In [ ]:
# SAR VV backscatter evalscript
SAR_EVALSCRIPT = """
//VERSION=3
function setup() {
  return {
    input: [{
      bands: ["VV"],
      units: "DB"
    }],
    output: {
      bands: 1,
      sampleType: "FLOAT32"
    }
  };
}

function evaluatePixel(sample) {
  // Return VV backscatter in dB
  return [sample.VV];
}
"""

def request_sar_vv_layer(
    bbox: List[float],
    date: str,
    width: int = 256,
    height: int = 256,
) -> Tuple[Optional[Dict], Optional[bytes]]:
    """Request SAR VV backscatter from Process API."""
    print(f"🔍 Requesting SAR VV layer for {date}...")
    
    request_body = {
        "input": {
            "bounds": {
                "bbox": bbox,
                "properties": [
                    {"name": "DateTime", "operator": ">=", "value": f"{date}T00:00:00Z"},
                    {"name": "DateTime", "operator": "<", "value": f"{date}T23:59:59Z"},
                ],
            },
            "data": [
                {
                    "type": "sentinel-1-grd",
                    "dataFilter": {"resolution": "HIGH"},
                    "processing": {
                        "orthorectify": "true",
                        "backCoeff": "GAMMA0_TERRAIN",
                        "demInstance": "COPERNICUS_30"
                    }
                }
            ]
        },
        "evalscript": SAR_EVALSCRIPT,
        "output": {
            "width": width,
            "height": height,
            "responses": [{"identifier": "default", "format": {"type": "image/tiff"}}]
        }
    }
    
    try:
        headers = auth.get_headers()
        response = requests.post(PROCESS_API_URL, json=request_body, headers=headers, timeout=30)
        response.raise_for_status()
        
        image_data = response.content
        metadata = {
            "status_code": response.status_code,
            "content_type": response.headers.get("Content-Type"),
            "size_bytes": len(image_data),
        }
        
        print(f"✓ SAR VV layer received ({len(image_data)} bytes)")
        return metadata, image_data
    except Exception as e:
        print(f"✗ SAR request failed: {e}")
        return None, None

# Demo: Simulate SAR VV backscatter data
np.random.seed(43)
SAR_VV_SIMULATED = np.random.normal(
    STATION_CONFIG["baseline_vv_db"]["mean"],
    STATION_CONFIG["baseline_vv_db"]["std"],
    (256, 256)
)

print("\n✓ SAR VV simulation created")
print(f"  Mean VV (dB): {SAR_VV_SIMULATED.mean():.1f}")
print(f"  Std Dev: {SAR_VV_SIMULATED.std():.1f}")
print(f"  Range: [{SAR_VV_SIMULATED.min():.1f}, {SAR_VV_SIMULATED.max():.1f}] dB")

## Section 6: Process API - Request Sentinel-1 SAR Backscatter (Cloud Fallback)

Request VV backscatter from Sentinel-1 GRD with orthorectification and GAMMA0_TERRAIN normalization.
SAR is weather-independent and can detect water anomalies when S2 is obscured by clouds.

In [ ]:
# Sentinel Hub evalscript for NDWI calculation
NDWI_EVALSCRIPT = """
//VERSION=3
function setup() {
  return {
    input: [{
      bands: ["B8A", "B11"],
      units: "reflectance"
    }],
    output: {
      bands: 1,
      sampleType: "FLOAT32"
    }
  };
}

function evaluatePixel(sample) {
  // NDWI = (B8A - B11) / (B8A + B11)
  // B8A: NIR (865nm), B11: SWIR (1610nm)
  let ndwi = (sample.B8A - sample.B11) / (sample.B8A + sample.B11);
  return [ndwi];
}
"""

def request_ndwi_layer(
    bbox: List[float],
    date: str,
    width: int = 256,
    height: int = 256,
) -> Tuple[Optional[Dict], Optional[bytes]]:
    """Request NDWI from Process API."""
    print(f"🔍 Requesting NDWI layer for {date}...")
    
    request_body = {
        "input": {
            "bounds": {
                "bbox": bbox,
                "properties": [
                    {"name": "DateTime", "operator": ">=", "value": f"{date}T00:00:00Z"},
                    {"name": "DateTime", "operator": "<", "value": f"{date}T23:59:59Z"},
                ],
                "filter": "eo:cloud_cover < 50"
            },
            "data": [
                {
                    "type": "sentinel-2-l2a",
                    "dataFilter": {"maxCloudCoverage": 50}
                }
            ]
        },
        "evalscript": NDWI_EVALSCRIPT,
        "output": {
            "width": width,
            "height": height,
            "responses": [{"identifier": "default", "format": {"type": "image/tiff"}}]
        }
    }
    
    try:
        headers = auth.get_headers()
        response = requests.post(PROCESS_API_URL, json=request_body, headers=headers, timeout=30)
        response.raise_for_status()
        
        image_data = response.content
        metadata = {
            "status_code": response.status_code,
            "content_type": response.headers.get("Content-Type"),
            "size_bytes": len(image_data),
        }
        
        print(f"✓ NDWI layer received ({len(image_data)} bytes)")
        return metadata, image_data
    except Exception as e:
        print(f"✗ NDWI request failed: {e}")
        return None, None

# Demo: Simulate NDWI data for analysis
# In production, use actual Process API response
np.random.seed(42)
NDWI_SIMULATED = np.random.normal(
    STATION_CONFIG["baseline_ndvi"]["mean"],
    STATION_CONFIG["baseline_ndvi"]["std"],
    (256, 256)
)

print("\n✓ NDWI simulation created")
print(f"  Mean NDWI: {NDWI_SIMULATED.mean():.3f}")
print(f"  Std Dev: {NDWI_SIMULATED.std():.3f}")
print(f"  Range: [{NDWI_SIMULATED.min():.3f}, {NDWI_SIMULATED.max():.3f}]")

## Section 5: Process API - Request Sentinel-2 NDWI Layer

Calculate Normalized Difference Water Index (NDWI) from Sentinel-2 L2A NIR and SWIR bands.
NDWI = (B8A - B11) / (B8A + B11), where B8A is NIR (865nm) and B11 is SWIR (1610nm).

In [ ]:
def get_date_range(days_back: int = 7) -> Tuple[str, str]:
    """Get ISO 8601 date range for last N days."""
    end_date = datetime.utcnow()
    start_date = end_date - timedelta(days=days_back)
    return start_date.strftime("%Y-%m-%d"), end_date.strftime("%Y-%m-%d")

def search_sentinel2_catalog(
    bbox: List[float],
    start_date: str,
    end_date: str,
    max_cloud_cover: float = 80.0,
    limit: int = 10,
) -> Dict[str, Any]:
    """
    Search Sentinel-2 L2A products via Catalog API.
    
    Args:
        bbox: [min_lon, min_lat, max_lon, max_lat]
        start_date: ISO 8601 date
        end_date: ISO 8601 date
        max_cloud_cover: Max cloud coverage %
        limit: Max results
    """
    print(f"📡 Searching Sentinel-2 L2A catalog ({start_date} to {end_date})...")
    
    query = {
        "type": "SearchProperties",
        "bbox": bbox,
        "datetime": f"{start_date}T00:00:00Z/{end_date}T23:59:59Z",
        "collections": ["sentinel-2-l2a"],
        "filter-lang": "cql2-text",
        "filter": f"eo:cloud_cover < {max_cloud_cover}",
        "limit": limit,
        "sortby": [{"field": "properties.datetime", "direction": "desc"}],
    }
    
    try:
        headers = auth.get_headers()
        response = requests.post(CATALOG_API_URL, json=query, headers=headers, timeout=15)
        response.raise_for_status()
        results = response.json()
        print(f"✓ Found {len(results.get('features', []))} Sentinel-2 products")
        return results
    except Exception as e:
        print(f"✗ Catalog search failed: {e}")
        return {"features": []}

def search_sentinel1_catalog(
    bbox: List[float],
    start_date: str,
    end_date: str,
    limit: int = 10,
) -> Dict[str, Any]:
    """Search Sentinel-1 GRD products via Catalog API."""
    print(f"📡 Searching Sentinel-1 GRD catalog ({start_date} to {end_date})...")
    
    query = {
        "type": "SearchProperties",
        "bbox": bbox,
        "datetime": f"{start_date}T00:00:00Z/{end_date}T23:59:59Z",
        "collections": ["sentinel-1-grd"],
        "limit": limit,
        "sortby": [{"field": "properties.datetime", "direction": "desc"}],
    }
    
    try:
        headers = auth.get_headers()
        response = requests.post(CATALOG_API_URL, json=query, headers=headers, timeout=15)
        response.raise_for_status()
        results = response.json()
        print(f"✓ Found {len(results.get('features', []))} Sentinel-1 products")
        return results
    except Exception as e:
        print(f"✗ Catalog search failed: {e}")
        return {"features": []}

# Execute catalog search
start_date, end_date = get_date_range(days_back=7)
print(f"\n📅 Searching for products from {start_date} to {end_date}")

if COPERNICUS_CLIENT_ID != "YOUR_CLIENT_ID_HERE":
    s2_results = search_sentinel2_catalog(
        bbox=STATION_CONFIG["bbox"],
        start_date=start_date,
        end_date=end_date,
        max_cloud_cover=80,
        limit=5,
    )
    
    s1_results = search_sentinel1_catalog(
        bbox=STATION_CONFIG["bbox"],
        start_date=start_date,
        end_date=end_date,
        limit=5,
    )
    
    print("\n📊 Sentinel-2 L2A Products:")
    for feature in s2_results.get("features", [])[:3]:
        props = feature.get("properties", {})
        print(f"  • {feature.get('id')}")
        print(f"    Date: {props.get('datetime')}, Cloud: {props.get('eo:cloud_cover', 'N/A')}%")
    
    print("\n📊 Sentinel-1 GRD Products:")
    for feature in s1_results.get("features", [])[:3]:
        props = feature.get("properties", {})
        print(f"  • {feature.get('id')}")
        print(f"    Date: {props.get('datetime')}, Orbit: {props.get('sat:orbit_state', 'N/A')}")
else:
    print("\n⚠️ Skipping catalog search (using demo credentials)")

## Section 4: Catalog API - Search Sentinel-2 L2A and Sentinel-1 GRD Scenes

Query available products within AOI and date range, filter by cloud cover.

In [ ]:
# Define river stations for hackathon MVP
STATIONS = {
    "vardar_vs": {
        "name": "Vardar: Veles to Skopje",
        "river": "Vardar",
        "segment": "Veles-Skopje",
        "bbox": [21.45, 41.92, 21.65, 42.02],  # [min_lon, min_lat, max_lon, max_lat]
        "baseline_ndvi": {"mean": 0.35, "std": 0.12},
        "baseline_vv_db": {"mean": -11.5, "std": 2.1},
        "flow_speed_mps": 1.2,
        "downstream_targets": ["skopje_city", "vardar_mouth"],
    },
    "treska_km": {
        "name": "Treska: Kozjak to Matka",
        "river": "Treska",
        "segment": "Kozjak-Matka",
        "bbox": [20.95, 41.92, 21.15, 42.08],
        "baseline_ndvi": {"mean": 0.38, "std": 0.14},
        "baseline_vv_db": {"mean": -10.8, "std": 2.3},
        "flow_speed_mps": 0.95,
        "downstream_targets": ["vardar_vs"],
    },
    "ohrid_s": {
        "name": "Ohrid Lake: Studencista",
        "river": "Ohrid Lake Inflow",
        "segment": "Studencista Spring",
        "bbox": [20.78, 41.09, 20.88, 41.19],
        "baseline_ndvi": {"mean": 0.42, "std": 0.10},
        "baseline_vv_db": {"mean": -11.2, "std": 1.9},
        "flow_speed_mps": 0.5,
        "downstream_targets": ["ohrid_city"],
    },
}

# Select active station for this analysis
ACTIVE_STATION_ID = "vardar_vs"
STATION_CONFIG = STATIONS[ACTIVE_STATION_ID]

print(f"🔵 Active Station: {STATION_CONFIG['name']}")
print(f"   Bounding Box: {STATION_CONFIG['bbox']}")
print(f"   River: {STATION_CONFIG['river']}")
print(f"   Baseline NDVI: μ={STATION_CONFIG['baseline_ndvi']['mean']:.2f}, σ={STATION_CONFIG['baseline_ndvi']['std']:.2f}")
print(f"   Baseline VV (SAR): μ={STATION_CONFIG['baseline_vv_db']['mean']:.1f} dB, σ={STATION_CONFIG['baseline_vv_db']['std']:.1f} dB")

## Section 3: Define River Stations and AOIs

Configure hardcoded areas of interest (AOIs) for hackathon MVP.

In [ ]:
# Replace with your actual credentials from https://dataspace.copernicus.eu/
COPERNICUS_CLIENT_ID = "YOUR_CLIENT_ID_HERE"
COPERNICUS_CLIENT_SECRET = "YOUR_CLIENT_SECRET_HERE"

# Endpoints
OAUTH_TOKEN_URL = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
CATALOG_API_URL = "https://sh.dataspace.copernicus.eu/api/v1/catalog/1.0.0/search"
PROCESS_API_URL = "https://sh.dataspace.copernicus.eu/api/v1/process"

class CopernicusAuthManager:
    """OAuth2 client_credentials authentication."""
    
    def __init__(self, client_id: str, client_secret: str):
        self.client_id = client_id
        self.client_secret = client_secret
        self.access_token = None
        self.token_expires_at = None
    
    def get_token(self) -> str:
        """Get or refresh OAuth2 access token."""
        if self.access_token and self.token_expires_at and time.time() < self.token_expires_at:
            return self.access_token
        
        print("📍 Requesting OAuth2 token from Copernicus Data Space...")
        
        auth = (self.client_id, self.client_secret)
        payload = {"grant_type": "client_credentials"}
        
        response = requests.post(OAUTH_TOKEN_URL, auth=auth, data=payload, timeout=10)
        response.raise_for_status()
        token_data = response.json()
        
        self.access_token = token_data["access_token"]
        expires_in = token_data.get("expires_in", 3600)
        self.token_expires_at = time.time() + (expires_in * 0.95)
        
        print(f"✓ Token obtained (expires in {expires_in}s)")
        return self.access_token
    
    def get_headers(self) -> Dict[str, str]:
        """Get headers with bearer token."""
        token = self.get_token()
        return {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json",
        }

# Initialize auth manager
auth = CopernicusAuthManager(COPERNICUS_CLIENT_ID, COPERNICUS_CLIENT_SECRET)

# Test authentication (will fail with demo credentials)
if COPERNICUS_CLIENT_ID != "YOUR_CLIENT_ID_HERE":
    try:
        headers = auth.get_headers()
        print("✓ Authentication successful")
    except Exception as e:
        print(f"⚠️ Authentication failed (expected with demo credentials): {e}")

## Section 2: OAuth2 Authentication to Copernicus Data Space

Authenticate using client_credentials flow to obtain access token for API requests.

In [ ]:
import requests
import json
import numpy as np
import time
from datetime import datetime, timedelta
from typing import Dict, Any, Optional, Tuple, List
import base64

print("✓ Core libraries imported successfully")

## Section 1: Setup and Imports

Import required libraries for API communication, data processing, and visualization.

# WaterWatch: Connected Copernicus Sentinel Hub Pipeline

**Goal:** Unified water anomaly detection using satellite data fusion (Sentinel-2 NDWI + Sentinel-1 SAR) with citizen reporting and downstream risk propagation.

**Key Features:**
- OAuth2 authentication to Copernicus Data Space
- Catalog API search for S2 L2A and S1 GRD scenes
- NDWI calculation for optical water index
- SAR backscatter fallback (cloud-independent)
- Statistical risk signal (z-score based)
- Citizen report integration
- Downstream ETA prediction
- Production-ready alert JSON

**AOIs (Hackathon MVP):**
- Vardar: Veles to Skopje
- Treska: Kozjak to Matka
- Ohrid: Studencista Spring

**References:**
- [Copernicus Data Space Hub](https://dataspace.copernicus.eu/)
- [Sentinel Hub Process API](https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/)
- [OpenID Connect](https://identity.dataspace.copernicus.eu/)